In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

import torch
import matplotlib.pyplot as plt
import numpy as np

import sys
sys.path.insert(0, "../../src")

from juart.conopt.functional.fourier import (
    fourier_transform_adjoint,
    fourier_transform_forward,
    nonuniform_fourier_transform_adjoint,
)

from juart.vis.interactive import InteractiveMultiPlotter3D

In [ ]:
device = "cuda:2"
image = torch.load("../../../images/ResNet_0i_100DC_10P_1000spk_normalized/model_epoch_15").to(device).cpu().abs().numpy()[:,:,:,0,0]
image /= np.abs(image[:,:,64]).max()

In [ ]:
InteractiveMultiPlotter3D([image],
                          layout = [1,1],
                          title = None,
                          cmap="gray",
                          vmin = 0,
                          vmax = 1,
                          compact_plotting = True,
                          show_axis = False,
                          activate_colorbar = False).interactive

In [ ]:
FT = fourier_transform_forward(torch.from_numpy(image), [0,1,2])
print(image.shape)

In [ ]:
InteractiveMultiPlotter3D([FT.real,FT.imag],
                          layout = [1,2],
                          title = None,
                          cmap="gray",
                          vmin = 0,
                          vmax = 1,
                          compact_plotting = True,
                          show_axis = False,
                          activate_colorbar = False).interactive

In [ ]:
coords = torch.stack(torch.meshgrid(
    torch.arange(FT.shape[0]),
    torch.arange(FT.shape[0]),
    torch.arange(FT.shape[0]),
    indexing='ij'
), dim=-1)

center = (FT.shape[0]) / 2.0
dist = torch.sqrt(torch.sum((coords - center)**2, dim=-1))

mask = (dist <= 40).float()

FT_masked = FT * mask

In [ ]:
low_pass_image = fourier_transform_adjoint(FT_masked, [0,1,2])

In [ ]:
InteractiveMultiPlotter3D([low_pass_image.abs(), FT_masked.abs()],
                          layout = [1,1],
                          title = ["low_pass_image","masked kspace"],
                          cmap="gray",
                          vmin = 0,
                          vmax = 1,
                          compact_plotting = False,
                          show_axis = False,
                          activate_colorbar = False).interactive